# MiroFish — narrative-simulation explainability

The MiroFish layer (modes A/B/C) over SentiSense: a multi-agent crowd simulation that, per
trading day, seeds on news ≤T and forecasts the next-day TA-125 narrative. This notebook
reads the **cached** sim artifacts (`narrative_sim` / `_graph` / `_report`) — it does not
re-run the sims — and shows: coverage, event-study reports + the **agent graph**, whether
the sim consensus aligns with reality, and the **with-sim ablation** vs XGBoost/LSTM/TimesFM.

Prereqs: `uv sync --extra ml --extra finance --extra notebook --extra miro`; sims cached
(`scripts/run_miro_window.py`). The ablation cell (§4) is heavy (re-runs the models).


## 0. Setup

In [ ]:
import importlib, subprocess, sys
for pkg, imp in [("networkx","networkx"), ("seaborn","seaborn")]:
    if importlib.util.find_spec(imp) is None:
        subprocess.run([sys.executable,"-m","pip","install","-q",pkg], check=False)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sqlalchemy import text
import sentisense
from sentisense.db import get_engine
from sentisense.sim.graph_api import graph_for_date, latest_graph, report_for_date
sns.set_theme(style="whitegrid"); engine = get_engine()


## 1. Sim coverage
How many causal day-sims are cached, and the daily consensus (`dir_score`) over time.

In [ ]:
sim = pd.read_sql(text('''
    SELECT sim_date::date AS date, AVG(dir_score) dir, AVG(confidence) conf,
           AVG(disagreement) disag, COUNT(*) seeds, AVG(n_agents) agents
    FROM narrative_sim GROUP BY sim_date ORDER BY sim_date'''), engine, parse_dates=["date"])
print("cached sim-days:", len(sim))
if len(sim):
    sim = sim.set_index("date")
    fig, ax = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
    ax[0].bar(sim.index, sim["seeds"], width=1.0); ax[0].set_title("Seeds per day (sim coverage)")
    ax[1].plot(sim.index, sim["dir"], lw=0.9, color="#7c3aed"); ax[1].axhline(0, ls=":", c="gray")
    ax[1].set_title("Daily agent consensus dir_score (bull>0 / bear<0)"); plt.show()
    display(sim.describe())
else:
    print("No sims cached yet — run scripts/run_miro_window.py.")


## 2. Event study + the agent graph (mode B)
Pick a cached date → its qualitative report + the simulated agent/knowledge graph.

In [ ]:
import networkx as nx
EVENT_DATE = (sim.index.max().date().isoformat() if len(sim) else None)   # default: latest cached
print("event date:", EVENT_DATE)
rep = report_for_date(EVENT_DATE, engine) if EVENT_DATE else None
if rep:
    print("=== report (first 1500 chars) ==="); print((rep["report_md"] or "")[:1500])
g = graph_for_date(EVENT_DATE, engine) if EVENT_DATE else None
if g and g["nodes"]:
    G = nx.DiGraph()
    for n in g["nodes"]: G.add_node(n["id"], label=n.get("label", n["id"]), type=n.get("type"))
    for e in g["edges"]: G.add_edge(e["src"], e["dst"], weight=e.get("weight", 1.0))
    fig, ax = plt.subplots(figsize=(10, 8))
    pos = nx.spring_layout(G, seed=0, k=0.6)
    nx.draw_networkx_nodes(G, pos, node_size=120, node_color="#3b82f6", alpha=0.7, ax=ax)
    nx.draw_networkx_edges(G, pos, alpha=0.25, arrows=False, ax=ax)
    big = sorted(G.degree, key=lambda x: x[1], reverse=True)[:15]
    nx.draw_networkx_labels(G, pos, labels={n: G.nodes[n].get("label","")[:18] for n,_ in big}, font_size=8, ax=ax)
    ax.set_title(f"MiroFish agent/knowledge graph — {EVENT_DATE} ({g['meta']})"); ax.axis("off"); plt.show()
else:
    print("No graph cached for that date.")


## 3. Does the crowd consensus align with reality?
Sim `dir_score` (≤T, causal) vs the realised next-day TA-125 direction. AUC≈0.5 ⇒ no edge.

In [ ]:
from sentisense.models.backtest import directions_from_price, forecast_to_proba, direction_metrics
from sentisense.constants import TA125_CSV
ta = pd.read_csv(TA125_CSV); ta["Date"]=pd.to_datetime(ta["Date"],errors="coerce")
price = ta.dropna(subset=["Date"]).set_index("Date").sort_index()["Price"].astype(str).str.replace(",","",regex=False).astype(float)
truth = directions_from_price(price)
if len(sim):
    common = sim.index.intersection(truth.index)
    if len(common) > 5:
        sc = forecast_to_proba(sim.loc[common, "dir"].to_numpy())
        m = direction_metrics(sc, truth.loc[common].to_numpy())
        print(f"sim-consensus vs next-day direction over {len(common)} days:",
              {k: round(v,4) for k,v in m.items()})
    else:
        print("too few overlapping days for a metric.")


## 4. With-sim ablation (mode A)
Reuses `pipeline_compare.build_leaderboard(with_sim=True)` — XGBoost/LSTM/TimesFM with vs without the sim layer, both regimes. **Heavy** (re-runs the models / needs the tuned studies).

In [ ]:
import sys; sys.path.insert(0, "scripts")
from pipeline_compare import build_leaderboard, _to_markdown
board = build_leaderboard(["CUT","FULL"], use_timesfm=True, with_sim=True, xgb_trials=20)
display(board.round(4))
print(_to_markdown(board))


## 5. Notes
- §3 is the honest test: if the causal sim consensus has AUC≈0.5 vs next-day direction, the
  crowd sim adds no 1-day edge (expected under EMH) — its value is the event-study narrative
  (§2) + the agent graph for the UI.
- §4 measures whether `+sim` rows beat their no-sim twins out-of-sample. Compare
  `XGBoost` vs `XGBoost+sim`, `LSTM` vs `LSTM+sim`, `TimesFM-tuned` vs `TimesFM-tuned+sim`.
- All sim artifacts are cached (no live MiroFish needed to read this notebook); the agent
  graph follows docs/miro/UI_GRAPH_CONTRACT.md.
